In [2]:
import pandas as pd 
import os
import json

# Cleaning and Uploading Dataset

## Functions

### Renaming Files

In [3]:
def rename_files(path, prefix):
    """
    Renames image files and their associated .json and .txt files
    in the given directory using a sequential prefix-based naming scheme.

    Args:
        path (str):   Path to the directory containing the files.
        prefix (str): Prefix name for the renamed files (e.g., 'nightlife').
    """
    img_extensions = ('.jpg', '.png', '.webp', '.jpeg')

    # Get and sort image files for consistent ordering
    img_files = sorted([f for f in os.listdir(path) if f.endswith(img_extensions)])

    for i, img_name in enumerate(img_files, start=1):
        img_ext = os.path.splitext(img_name)[1]  # e.g., '.jpg'

        # Old paths
        old_img  = os.path.join(path, img_name)
        old_json = os.path.join(path, img_name + '.json')
        old_txt  = os.path.join(path, img_name + '.txt')

        # New names
        new_base = f'{prefix}_{i}{img_ext}'      
        new_img  = os.path.join(path, new_base)
        new_json = os.path.join(path, new_base + '.json')
        new_txt  = os.path.join(path, new_base + '.txt')

        if os.path.exists(old_img):
            os.rename(old_img, new_img)
        if os.path.exists(old_json):
            os.rename(old_json, new_json)
        if os.path.exists(old_txt):
            os.rename(old_txt, new_txt)


### Labeling Files (Unsafe/Safe)

Dilakukan di python bukan di ipynb karena cv2 tidak akan muncul di ipynb

In [ ]:
def label_files(path, unsafe_path, safe_path):
    """
    Label file berdasarkan safe/unsafe lalu simpan ke dalam 
    sebuah folder yang berbeda.

    Args:
        path (str): File path asli.
        unsafe_path (str): File path untuk menempatkan unsafe images.
        safe_path(str): File path untuk menempatkan safe images.
    
    Controls:
        [u] = Unsafe
        [s] = Safe
        [q] = Quit
    """

    # Buat kedua folder jika tidak ada
    os.makedirs(unsafe_path, exist_ok=True)  
    os.makedirs(safe_path, exist_ok=True)    

    for filename in os.listdir(path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.webp')):
            img_path = os.path.join(path, filename)
            img = cv2.imread(img_path)

            display_img = cv2.resize(img, (500, 500))
            cv2.imshow('Labeling - [u] Unsafe  [s] Safe  [q] Quit', display_img)

            key = cv2.waitKey(0)

            if key == ord('u'):
                shutil.move(img_path, os.path.join(unsafe_path, filename))
                print(f"[UNSAFE] {filename}")
            elif key == ord('s'):
                shutil.move(img_path, os.path.join(safe_path, filename)) 
                print(f"[SAFE]   {filename}")
            elif key == ord('q'):
                print("Quit labeling.")
                break

    cv2.destroyAllWindows()

### Creating DataFrame

In [4]:
def build_dataframe(tag_name, violation_type):
    """
    Membuat dataframe dari image yang sudah dilabeli agar sesuai dengan dataframe di huggingface

    Keterangan:
        sample_name    - nama file
        description    - untuk diisi image captioning
        category       - 'safe' atau 'unsafe'
        violation_type - jenis pelanggaran
        type           - 'image'
        link           - post url dari json
        image          - gambar

    Args:
        tag_name (str):       dataset folder
        violation_type (str): violation type

    Returns:
        pd.DataFrame
    """
    img_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    records = []

    for category in ['safe', 'unsafe']:
        if category == 'safe':
            violation = None
        elif category == 'unsafe':
            violation = violation_type
        
        folder_path = os.path.join(tag_name, category)

        if not os.path.exists(folder_path):
            print(f"Folder not found, skipping: {folder_path}")
            continue

        for filename in sorted(os.listdir(folder_path)):
            if not filename.endswith(img_extensions):
                continue

            img_path  = os.path.join(folder_path, filename)

            # Look for the matching .json sidecar file (same folder as image)
            # Original scrape keeps .json next to the images in gallery-dl output,
            # but after labeling the images move — so check current folder first.
            base_path = 'ig_metadata'
            json_path = os.path.join(base_path, tag_name, filename) + '.json'
            post_url  = None

            if os.path.exists(json_path):
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    post_url = data.get('post_url', None)

            records.append({
                'sample_name': filename,
                'description': None,
                'category': category,
                'violation_type': violation,
                'type': 'image',
                'link': post_url,
                'image': img_path
            })

    df = pd.DataFrame(records)
    return df


### Upload to HF

In [ ]:
from datasets import Dataset,Image, concatenate_datasets, load_dataset
from huggingface_hub import login

In [ ]:
from datasets import Dataset, Image
from huggingface_hub import login

def upload_to_huggingface(df, repo_name, hf_token):
    """
    Convert DataFrame to Datasets

    Args:
        df         (pd.DataFrame): DataFrame from build_dataset()
        repo_name  (str):          e.g. 'your-username/your-dataset-name'
        hf_token   (str):          Your HuggingFace token (Write permission)
    """
    login(token=hf_token)

    new_dataset = Dataset.from_pandas(df, preserve_index=False)
    new_dataset = new_dataset.cast_column("image", Image())

    existing_dataset = load_dataset(repo_name, split='train')
    combined_dataset = concatenate_dataset([existing_dataset, new_dataset])

    combined_dataset.push_to_hub(repo_name, split="train", append=True)


### Path to Image (another HF approach)

In [5]:
from PIL import Image
import io

In [6]:
def path_to_bytes(img_path):
    """
    Baca path dan convert ke bytes
    """
    try:
        with Image.open(img_path) as img:
            buf = io.BytesIO()
            img.save(buf, format=img.format or "WEBP")
            return buf.getvalue()
    except Exception as e:
        print(f"Failed: {img_path} → {e}")
        return None

### Check Duplicate Images

In [15]:
import os
import hashlib
from collections import defaultdict

def get_file_hash(filepath, chunk_size=8192):
    """
    Baca file per chunk bytes dan hitung hash SHA256.
    """
    hasher = hashlib.sha256()
    with open(filepath, 'rb') as f:         
        while chunk := f.read(chunk_size):   
            hasher.update(chunk)
    return hasher.hexdigest()

def check_duplicate(unsafe_path=None, safe_path=None, label_type="no", path=None):
    """
    Cari gambar yang duplikat dalam file path (byte-by-byte via hash)
    """
    def is_image(f):
        return f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))

    if label_type == "yes":
        unsafe_files = [os.path.join(unsafe_path, f) for f in os.listdir(unsafe_path) if is_image(f)]
        safe_files   = [os.path.join(safe_path, f)   for f in os.listdir(safe_path)   if is_image(f)]
        all_files = unsafe_files + safe_files
    elif label_type == "no":
        all_files = [os.path.join(path, f) for f in os.listdir(path) if is_image(f)]

    hash_map = defaultdict(list)

    for filepath in all_files:
        if os.path.isfile(filepath):
            file_hash = get_file_hash(filepath)
            hash_map[file_hash].append(filepath)

    duplicates = {h: paths for h, paths in hash_map.items() if len(paths) > 1}

    if duplicates:
        print(f"Found {len(duplicates)} duplicate group(s):")
        for h, paths in duplicates.items():
            print(f"\n  Hash: {h[:12]}...")
            for p in paths:
                print(f"    - {p}")
    else:
        print("No duplicates found.")

    return duplicates



def remove_duplicates(duplicates):
    if not duplicates:
        print("No duplicates")
        return
    
    total_remove = 0
    for h, paths in duplicates.items():
        # simpan file pertama
        keep = paths[0]
        to_delete = paths[1:]

        for p in to_delete:
            try: 
                os.remove(p)
                print(f"File deleted: {p}")
                total_remove += 1
            except Exception as e:
                print(f"Fail: {e}")
                

## Implementation

In [7]:
import uuid
from dotenv import load_dotenv
HF_TOKEN = os.getenv("HF_TOKEN")

### indoclubbing

In [8]:
ic_dupe = check_duplicate('indoclubbing/unsafe', 'indoclubbing/safe', label_type="yes")

No duplicates found.


In [23]:
remove_duplicates(ic_dupe)

File deleted: indoclubbing/unsafe\indoclubbing_153.jpg
File deleted: indoclubbing/unsafe\indoclubbing_199.webp
File deleted: indoclubbing/unsafe\indoclubbing_62.jpg


In [9]:
df_indoclubbing = build_dataframe('indoclubbing', 'adult_lifestyle')

In [11]:
df_indoclubbing["image"] = df_indoclubbing["image"].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_indoclubbing.to_parquet(shard_name)

In [ ]:
df_ic_export = df_indoclubbing.drop(columns=['image'])
df_ic_export.to_csv('indoclubbing_metadata.csv', index=False)

### jomok

In [ ]:
rename_files('gallery-dl/instagram/tag/jomok', 'jomok')

In [12]:
j_dupe = check_duplicate('jomok/unsafe', 'jomok/safe', label_type="yes")

No duplicates found.


In [13]:
df_jomok = build_dataframe('jomok', 'inappropriate_humor')

In [14]:
df_jomok["image"] = df_jomok["image"].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_jomok.to_parquet(shard_name)

In [52]:
df_j_export = df_jomok.drop(columns=['image'])
df_j_export.to_csv('jomok_metadata.csv', index=False)

### darkjokes (not yet cleaned, masih harus manual)


In [ ]:
rename_files('gallery-dl/instagram/tag/darkjokes', 'dark_jokes')

In [ ]:
df_darkjokes = build_dataframe('darkjokes', 'inappropriate_humor')
darkjokes_desc = pd.read_csv('darkjokes/label_results.csv')
df_darkjokes['description'] = darkjokes_desc['description']

In [ ]:
df_darkjokes["image"] = df_darkjokes["image"].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_jomok.to_parquet(shard_name)

### stemforkids

In [ ]:
rename_files('gallery-dl/instagram/tag/stemforkids', 'stemforkids')

In [15]:
stem_dupe = check_duplicate('stemforkids/unsafe', 'stemforkids/safe', label_type="yes")

No duplicates found.


In [18]:
remove_duplicates(stem_dupe)

File deleted: stemforkids/safe\stemforkids_10.jpg
File deleted: stemforkids/safe\stemforkids_11.jpg
File deleted: stemforkids/safe\stemforkids_12.jpg
File deleted: stemforkids/safe\stemforkids_13.jpg
File deleted: stemforkids/safe\stemforkids_14.jpg
File deleted: stemforkids/safe\stemforkids_15.jpg
File deleted: stemforkids/safe\stemforkids_16.jpg
File deleted: stemforkids/safe\stemforkids_17.jpg
File deleted: stemforkids/safe\stemforkids_18.jpg
File deleted: stemforkids/safe\stemforkids_19.jpg
File deleted: stemforkids/safe\stemforkids_2.jpg
File deleted: stemforkids/safe\stemforkids_20.jpg
File deleted: stemforkids/safe\stemforkids_21.jpg
File deleted: stemforkids/safe\stemforkids_22.jpg
File deleted: stemforkids/safe\stemforkids_25.jpg
File deleted: stemforkids/safe\stemforkids_26.jpg
File deleted: stemforkids/safe\stemforkids_27.jpg
File deleted: stemforkids/safe\stemforkids_28.jpg
File deleted: stemforkids/safe\stemforkids_37.jpg
File deleted: stemforkids/safe\stemforkids_38.jpg
F

In [16]:
df_sfk = build_dataframe('stemforkids', 'inappropriate_content')
sfk_desc = pd.read_csv('stemforkids/label_results.csv')
df_sfk['description'] = sfk_desc['description']

In [17]:
df_sfk['image'] = df_sfk['image'].apply(path_to_bytes)
shard_name = f"train-{uuid.uuid4().hex[:8]}.parquet"
df_sfk.to_parquet(shard_name)

In [53]:
df_sfk_export = df_sfk.drop(columns=['image'])
df_sfk_export.to_csv('stemforkids_metadata.csv', index=False)

### lgbt

In [ ]:
rename_files('gallery-dl/instagram/tag/lgbt', 'lgbt')

### missile

In [8]:
rename_files('gallery-dl/instagram/tag/missile', 'missile')

In [16]:
mis_dupe = check_duplicate(path='gallery-dl/instagram/tag/missile', label_type="no")

Found 4 duplicate group(s):

  Hash: 041fb436f562...
    - gallery-dl/instagram/tag/missile\missile_101.jpg
    - gallery-dl/instagram/tag/missile\missile_111.jpg
    - gallery-dl/instagram/tag/missile\missile_119.jpg
    - gallery-dl/instagram/tag/missile\missile_219.jpg

  Hash: 27878fb87feb...
    - gallery-dl/instagram/tag/missile\missile_47.jpg
    - gallery-dl/instagram/tag/missile\missile_48.jpg

  Hash: fd700e7917ef...
    - gallery-dl/instagram/tag/missile\missile_51.jpg
    - gallery-dl/instagram/tag/missile\missile_57.jpg

  Hash: d1e30f49330e...
    - gallery-dl/instagram/tag/missile\missile_91.jpg
    - gallery-dl/instagram/tag/missile\missile_93.jpg


In [17]:
remove_duplicates(mis_dupe)

File deleted: gallery-dl/instagram/tag/missile\missile_111.jpg
File deleted: gallery-dl/instagram/tag/missile\missile_119.jpg
File deleted: gallery-dl/instagram/tag/missile\missile_219.jpg
File deleted: gallery-dl/instagram/tag/missile\missile_48.jpg
File deleted: gallery-dl/instagram/tag/missile\missile_57.jpg
File deleted: gallery-dl/instagram/tag/missile\missile_93.jpg
